In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE = "/content/drive/MyDrive/rl-final-project"
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
import os, random, time, pickle
from collections import deque, namedtuple
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt
from PIL import Image
import imageio
import minatar

SEED = 1234
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
DRIVE = "/content/drive/MyDrive/rl-final-project"
os.makedirs(os.path.join(DRIVE, "visuals", "shaped_rollout"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "dqn_cnn"), exist_ok=True)
os.makedirs(os.path.join(DRIVE, "plots", "dqn_cnn"), exist_ok=True)

In [ ]:
# Shaped environment wrapper (inherits idea от MiniAtariStack)
class ShapedMiniAtariStack:
    def __init__(self, game_name="breakout", stack=4):
        self.env = minatar.Environment(game_name)
        self.stack = stack
        self.n_actions = self.env.num_actions()
        ss = self.env.state_shape()
        if isinstance(ss, int):
            self.h = self.w = int(np.sqrt(ss))
        elif len(ss) == 1:
            self.h = self.w = int(np.sqrt(ss[0]))
        else:
            self.h, self.w = ss[0], ss[1]
        self.frames = deque(maxlen=stack)
        # bookkeeping for shaping
        self.prev_brick_count = None
        self.prev_ball_y = None

    def reset(self):
        self.env.reset()
        st = self.env.state()
        f = self._single_preproc(st)
        self.frames.clear()
        for _ in range(self.stack):
            self.frames.append(f.copy())
        # init shaping bookkeeping
        self.prev_brick_count = self._count_bricks(st)
        self.prev_ball_y = self._ball_y(st)
        return self._get_state()

    def step(self, action):
        base_reward, done = self.env.act(int(action))  # returns (reward, done)
        st = self.env.state()
        f = self._single_preproc(st)
        self.frames.append(f.copy())
        shaped = self._shaping_signal(st)
        total_reward = float(base_reward) + float(shaped)
        return self._get_state(), float(total_reward), bool(done)

    def _single_preproc(self, st):
        arr = np.array(st, dtype=np.float32)
        if arr.ndim == 3:
            arr = arr.sum(axis=2)
        mn, mx = float(arr.min()), float(arr.max())
        rng = mx - mn if mx > mn else 1.0
        arr = (arr - mn) / rng
        return arr

    def _get_state(self):
        return np.stack(list(self.frames), axis=0).astype(np.float32)  # (C,H,W)

    def render_raw(self):
        raw = self.frames[-1]
        img = (raw * 255).astype(np.uint8)
        return img

    # ---------- shaping helpers ----------
    def _count_bricks(self, st):
        # Heuristic: one of channels corresponds to bricks — find channel with many True at init
        arr = np.array(st, dtype=np.int32)
        if arr.ndim == 3:
            counts = [int(np.sum(arr[:,:,c])) for c in range(arr.shape[2])]
            # choose channel with largest count (likely bricks)
            return counts[np.argmax(counts)]
        else:
            return 0

    def _ball_y(self, st):
        arr = np.array(st, dtype=np.float32)
        if arr.ndim == 3:
            # assume ball channel is the one with small sum (or detect via movement)
            # we'll detect ball by finding the channel with one or few nonzeros and return its centroid y
            sums = [np.sum(arr[:,:,c]) for c in range(arr.shape[2])]
            # choose channel with small but >0 sum
            cand = [c for c,s in enumerate(sums) if s>0 and s < max(sums)]
            if not cand:
                cand = [np.argmax(sums)]
            ch = cand[0]
            mat = arr[:,:,ch]
            ys = np.where(mat > 0)[0]
            if len(ys)==0:
                return None
            return float(np.mean(ys))  # row index ~ y
        else:
            return None

    def _shaping_signal(self, st):
        # Compute shaping value using two signals:
        # 1) brick destruction: +0.5 when brick count decreased
        # 2) ball moving upward: +0.05 when ball y decreased (smaller row index = up)
        s = 0.0
        # brick signal
        cur_bricks = self._count_bricks(st)
        if self.prev_brick_count is None:
            self.prev_brick_count = cur_bricks
        if cur_bricks < self.prev_brick_count:
            # bricks destroyed -> positive shaping
            s += 0.5 * (self.prev_brick_count - cur_bricks)  # proportional if multiple destroyed
        self.prev_brick_count = cur_bricks

        # ball motion signal
        cur_ball_y = self._ball_y(st)
        if cur_ball_y is None:
            cur_ball_y = self.prev_ball_y
        if self.prev_ball_y is None:
            self.prev_ball_y = cur_ball_y
        if (cur_ball_y is not None) and (self.prev_ball_y is not None):
            # moving up: current y < prev y (coordinates: 0 top)
            if cur_ball_y < self.prev_ball_y - 0.5:  # threshold to avoid noise
                s += 0.05
        self.prev_ball_y = cur_ball_y

        return s

In [ ]:
class DQNCNN(nn.Module):
    def __init__(self, in_channels=4, n_actions=6):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=0),
            nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*6*6, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )
    def forward(self, x):
        return self.fc(self.conv(x))

In [ ]:
Transition = namedtuple('Transition', ('state','action','reward','next_state','done'))
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state.copy(), int(action), float(reward), next_state.copy(), float(done)))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))
    def __len__(self):
        return len(self.buffer)

In [ ]:
def train_shaped_dqn_cnn(env, drive_dir,
                         num_steps=200000,
                         buffer_capacity=200000,
                         batch_size=32,
                         gamma=0.99,
                         lr=2.5e-4,
                         target_update_freq=5000,
                         start_learning=10000,
                         eps_start=1.0,
                         eps_final=0.05,
                         eps_decay=150000,
                         save_every=20000,
                         use_double=True):
    out_dir = os.path.join(drive_dir, "dqn_cnn")
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, "best_shaped_dqn_cnn.pth")
    expert_dataset_path = os.path.join(out_dir, "expert_dataset_shaped.pkl")
    rewards_plot = os.path.join(drive_dir, "plots", "dqn_cnn", "shaped_dqn_rewards.png")

    n_actions = env.n_actions
    net = DQNCNN(in_channels=env.stack, n_actions=n_actions).to(device)
    target = DQNCNN(in_channels=env.stack, n_actions=n_actions).to(device)
    target.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=lr)
    replay = ReplayBuffer(buffer_capacity)

    all_steps = 0
    episode_reward = 0.0
    episode_rewards = []
    best_avg = -float('inf')
    losses = []
    expert_transitions = []

    state = env.reset()
    while all_steps < num_steps:
        # eps schedule
        if all_steps >= eps_decay:
            eps = eps_final
        else:
            eps = eps_final + (eps_start - eps_final) * (1 - all_steps/eps_decay)

        if random.random() < eps:
            action = random.randrange(n_actions)
        else:
            with torch.no_grad():
                s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
                qvals = net(s)
                action = int(torch.argmax(qvals, dim=1).item())

        next_state, reward, done = env.step(action)  # reward already shaped inside env
        replay.push(state, action, reward, next_state, done)
        expert_transitions.append((state.copy(), int(action), float(reward), next_state.copy(), float(done)))

        episode_reward += reward
        all_steps += 1
        state = next_state

        if all_steps % 1000 == 0:
            episode_rewards.append(episode_reward)
            episode_reward = 0.0
            state = env.reset()

        if len(replay) > start_learning:
            batch = replay.sample(batch_size)
            states = np.stack(batch.state).astype(np.float32)
            next_states = np.stack(batch.next_state).astype(np.float32)
            actions = np.array(batch.action, dtype=np.int64)
            rewards = np.array(batch.reward, dtype=np.float32)
            dones = np.array(batch.done, dtype=np.float32)

            states_t = torch.tensor(states, dtype=torch.float32).to(device)
            next_states_t = torch.tensor(next_states, dtype=torch.float32).to(device)
            actions_t = torch.tensor(actions, dtype=torch.long).unsqueeze(1).to(device)
            rewards_t = torch.tensor(rewards, dtype=torch.float32).unsqueeze(1).to(device)
            dones_t = torch.tensor(dones, dtype=torch.float32).unsqueeze(1).to(device)

            q_values = net(states_t).gather(1, actions_t)
            with torch.no_grad():
                if use_double:
                    next_actions = torch.argmax(net(next_states_t), dim=1, keepdim=True)
                    q_next_target = target(next_states_t).gather(1, next_actions)
                else:
                    q_next_target = target(next_states_t).max(1)[0].unsqueeze(1)
                q_target = rewards_t + gamma * (1 - dones_t) * q_next_target

            loss = nn.functional.mse_loss(q_values, q_target)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 10.0)
            opt.step()
            losses.append(loss.item())

        if all_steps % target_update_freq == 0:
            target.load_state_dict(net.state_dict())

        if all_steps % save_every == 0:
            avg_recent = float(np.mean(episode_rewards[-50:])) if episode_rewards else 0.0
            print(f"[{time.strftime('%H:%M:%S')}] Step {all_steps}/{num_steps} eps={eps:.3f} avg_recent={avg_recent:.3f} replay_len={len(replay)}")
            if avg_recent > best_avg:
                best_avg = avg_recent
                torch.save(net.state_dict(), model_path)
                print("Saved improved shaped model ->", model_path)

    # final save
    torch.save(net.state_dict(), model_path)
    pickle.dump(expert_transitions, open(expert_dataset_path, "wb"))
    print("Training complete. model:", model_path)
    # plot
    if episode_rewards:
        plt.figure(figsize=(10,4))
        plt.plot(episode_rewards)
        plt.title("Shaped DQN-CNN episode rewards (chunks)")
        plt.grid(True)
        plt.savefig(rewards_plot, dpi=150)
        plt.close()

    return {"model_path": model_path, "expert_dataset_path": expert_dataset_path, "rewards_plot": rewards_plot}

In [ ]:
def save_shaped_rollout_gif(env, model_path, out_folder=os.path.join(DRIVE,"visuals","shaped_rollout"), episodes=3, steps=800, scale=3):
    net = DQNCNN(in_channels=env.stack, n_actions=env.n_actions).to(device)
    net.load_state_dict(torch.load(model_path, map_location=device))
    net.eval()
    os.makedirs(out_folder, exist_ok=True)
    all_scores = []
    for ep in range(episodes):
        s = env.reset()
        frames = []
        total = 0.0
        for t in range(steps):
            with torch.no_grad():
                x = torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)
                q = net(x)
                a = int(torch.argmax(q, dim=1).item())
            s, r, done = env.step(a)
            total += r
            raw = env.render_raw()
            # upscale for visibility
            im = Image.fromarray((raw).astype(np.uint8)).resize((200,200), Image.NEAREST)
            frames.append(np.array(im))
            if done:
                break
        all_scores.append(total)
        gif_path = os.path.join(out_folder, f"shaped_eval_ep{ep:02d}.gif")
        imageio.mimsave(gif_path, frames, fps=12)
        print("Saved", gif_path, "score", total)
    print("Mean score:", np.mean(all_scores), "std:", np.std(all_scores))
    return all_scores

# Usage after training:
# shaped_env = ShapedMiniAtariStack("breakout", stack=4)
# res = train_shaped_dqn_cnn(shaped_env, DRIVE, num_steps=150000, ...)
# save_shaped_rollout_gif(shaped_env, res['model_path'])